<a href="https://colab.research.google.com/github/Leonardozepeda04/edt-dataa-pipeline/blob/main/notebooks/tipos_seguros.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
#Crear dataset
url_tipos = "https://raw.githubusercontent.com/Leonardozepeda04/edt-dataa-pipeline/refs/heads/main/data/raw/tipos_seguro.csv"

In [3]:
tipos_seguro = pd.read_csv(url_tipos)

In [4]:
#Exploracion de datos
tipos_seguro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_tipo_seguro  12 non-null     int64 
 1   tipo            12 non-null     object
 2   categoria       10 non-null     object
 3   riesgo_base     10 non-null     object
dtypes: int64(1), object(3)
memory usage: 516.0+ bytes


In [5]:
#Limpieza de datos
def limpiar_dataframe(df):

    df.columns = df.columns.str.strip().str.lower()

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()

    df = df.replace(r'^\s*$', pd.NA, regex=True)

    df = df.drop_duplicates()

    return df

In [6]:
#Transformaciones
# Normalizar espacios y mayúsculas/minúsculas
tipos_seguro['tipo'] = tipos_seguro['tipo'].astype(str).str.strip().str.capitalize()
tipos_seguro['categoria'] = tipos_seguro['categoria'].astype(str).str.strip().str.capitalize()

# --- Limpieza de columna numérica ---
# Reemplazar comas por puntos y eliminar espacios
tipos_seguro['riesgo_base'] = tipos_seguro['riesgo_base'].astype(str).str.replace(",", ".").str.strip()

# Convertir a numérico (los valores no válidos se convierten en NaN)
tipos_seguro['riesgo_base'] = pd.to_numeric(tipos_seguro['riesgo_base'], errors='coerce')

# Por ejemplo, rellenar con 0 o con la media
tipos_seguro['riesgo_base'] = tipos_seguro['riesgo_base'].fillna(0)



In [7]:
# Ver resultado
print(tipos_seguro)

    id_tipo_seguro        tipo    categoria  riesgo_base
0                1        Pyme     Familiar         0.00
1                2  Industrial  Empresarial         4.68
2                3  Industrial     Familiar         5.10
3                4  Industrial     Personal         0.00
4                5        Auto  Empresarial         9.07
5                6  Industrial  Empresarial         2.52
6                7       Salud     Personal         0.92
7                8   Educación  Empresarial         7.42
8                9  Accidentes          Nan         5.68
9               10      Dental     Especial         2.70
10              11        Auto  Empresarial         4.33
11              12    Agrícola          Nan         0.00


In [8]:
# --- Separar válidos y rechazados ---
validos = tipos_seguro[
    tipos_seguro['tipo'].notna() &
    tipos_seguro['categoria'].notna() &
    tipos_seguro['riesgo_base'].notna()
].copy()

rechazados = tipos_seguro[
    tipos_seguro['tipo'].isna() |
    tipos_seguro['categoria'].isna() |
    tipos_seguro['riesgo_base'].isna()
].copy()



In [9]:
# Mostrar resultados
print("✅ Válidos:")
print(validos)

print("\n❌ Rechazados:")
print(rechazados)

✅ Válidos:
    id_tipo_seguro        tipo    categoria  riesgo_base
0                1        Pyme     Familiar         0.00
1                2  Industrial  Empresarial         4.68
2                3  Industrial     Familiar         5.10
3                4  Industrial     Personal         0.00
4                5        Auto  Empresarial         9.07
5                6  Industrial  Empresarial         2.52
6                7       Salud     Personal         0.92
7                8   Educación  Empresarial         7.42
8                9  Accidentes          Nan         5.68
9               10      Dental     Especial         2.70
10              11        Auto  Empresarial         4.33
11              12    Agrícola          Nan         0.00

❌ Rechazados:
Empty DataFrame
Columns: [id_tipo_seguro, tipo, categoria, riesgo_base]
Index: []


In [10]:
# --- Motivos de rechazo ---
def motivo(row):
    motivos = []
    if pd.isna(row['tipo']):
        motivos.append("tipo_vacio")
    if pd.isna(row['categoria']):
        motivos.append("categoria_vacia")
    if pd.isna(row['riesgo_base']):
        motivos.append("riesgo_base_vacio")
    return ",".join(motivos)

rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)

In [11]:
#Resultados
print("\n❌ Rechazados con motivos:")
print(rechazados[['id_tipo_seguro','tipo','categoria','riesgo_base','motivo_rechazo']])


❌ Rechazados con motivos:
Empty DataFrame
Columns: [id_tipo_seguro, tipo, categoria, riesgo_base, motivo_rechazo]
Index: []


In [12]:
#Exportar archivo curated
tipos_seguro.to_csv("tipos_seguro_curated.csv", index=False)